# 02 — RQ-KMeans Tokenization

Train and evaluate RQ-KMeans (hierarchical/sequential SID assignment from GRID).

In [1]:
DRY_RUN = True

In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.yambda_loader import load_embeddings
from src.tokenizers.rq_kmeans import RQKMeans
from src.evaluation.metrics import (
    reconstruction_mse, codebook_utilization, collision_rate, entropy_per_level
)

In [3]:
with open("../configs/default.yaml") as f:
    config = yaml.safe_load(f)

rq_cfg = config["rq_kmeans"]
suffix = "_dryrun" if DRY_RUN else ""

# Dry-run overrides
codebook_width = min(32, rq_cfg["codebook_width"]) if DRY_RUN else rq_cfg["codebook_width"]
num_hierarchies = min(4, rq_cfg["num_hierarchies"]) if DRY_RUN else rq_cfg["num_hierarchies"]
max_iter = 10 if DRY_RUN else rq_cfg["max_iter"]

print(f"codebook_width={codebook_width}, num_hierarchies={num_hierarchies}, max_iter={max_iter}")

codebook_width=32, num_hierarchies=4, max_iter=10


## Load Embeddings

In [4]:
embed_path = config["data"]["output_path"].replace(".parquet", f"{suffix}.parquet")
item_ids, embeddings = load_embeddings(embed_path)

Loaded 1000 embeddings of dim 64 from data/embeddings_dryrun.parquet


## Train RQ-KMeans

In [5]:
model = RQKMeans(
    num_hierarchies=num_hierarchies,
    codebook_width=codebook_width,
    normalize_residuals=rq_cfg["normalize_residuals"],
    max_iter=max_iter,
    mini_batch_size=rq_cfg["mini_batch_size"],
    device=rq_cfg["device"],
)
model.fit(embeddings)

RQ-KMeans levels: 100%|██████████| 4/4 [00:00<00:00, 34.16it/s]

RQ-KMeans fitted: 4 levels, codebook_width=32


## Encode & Evaluate

In [6]:
sids = model.encode(embeddings)
reconstructed = model.reconstruct(embeddings)

print(f"SIDs shape: {sids.shape}")
print(f"Reconstruction MSE: {reconstruction_mse(embeddings, reconstructed):.6f}")
print(f"Collision rate: {collision_rate(sids):.6f}")

SIDs shape: (1000, 4)
Reconstruction MSE: 0.009180
Collision rate: 0.000000


In [7]:
# Per-level stats
util = codebook_utilization(sids, codebook_width)
ent = entropy_per_level(sids, codebook_width)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
levels = np.arange(num_hierarchies)

axes[0].bar(levels, util, color="steelblue")
axes[0].set_title("Codebook Utilization per Level")
axes[0].set_xlabel("Level")
axes[0].set_ylabel("Utilization")
axes[0].set_ylim(0, 1.05)

axes[1].bar(levels, ent, color="steelblue")
axes[1].set_title("Normalized Entropy per Level")
axes[1].set_xlabel("Level")
axes[1].set_ylabel("Entropy")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

/var/folders/6t/0bc7rzr91pd3316c59fnhvwc0000gn/T/ipykernel_82324/1827907505.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# SID distribution histograms
fig, axes = plt.subplots(1, min(4, num_hierarchies), figsize=(14, 3))
if num_hierarchies == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.hist(sids[:, i], bins=codebook_width, edgecolor="black")
    ax.set_title(f"Level {i} codes")
    ax.set_xlabel("Code")
plt.tight_layout()
plt.show()

/var/folders/6t/0bc7rzr91pd3316c59fnhvwc0000gn/T/ipykernel_82324/1973411409.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Results

In [9]:
from pathlib import Path

# Save SIDs
sids_path = rq_cfg["sids_path"].replace(".parquet", f"{suffix}.parquet")
sids_df = pd.DataFrame({"item_id": item_ids})
for i in range(sids.shape[1]):
    sids_df[f"sid_{i}"] = sids[:, i]
Path(sids_path).parent.mkdir(parents=True, exist_ok=True)
sids_df.to_parquet(sids_path, index=False)
print(f"SIDs saved to {sids_path}")

# Save checkpoint
ckpt_path = rq_cfg["checkpoint_path"].replace(".pt", f"{suffix}.pt")
model.save(ckpt_path)

SIDs saved to outputs/rq_kmeans/sids_dryrun.parquet
RQ-KMeans saved to outputs/rq_kmeans/model_dryrun.pt
